# XThreat — Data Preprocessing Foundations (13-Step Learning Notebook)

Ei notebook ta **model training** er upor focus kore na. Eta **data preprocessing** er
proti-tar step tomake **haate kolome (from scratch)** dekhabe, tarpor sob shesh e
**sklearn `Pipeline`** diye ekshathe jora lagabe.

Domain: same **XThreat brute-force / credential-stuffing detection** dataset
(synthetic auth logs, per source-IP aggregated).

### 13 Steps
1. Missing Data
2. Cleaning
3. Outlier Handling
4. Sampling
5. Imbalance Handling
6. Transformation
7. Scaling
8. Encoding
9. Feature Engineering
10. Feature Selection
11. Simulation / Augmentation
12. Statistical Analysis
13. Extra Layer — Leakage / Train-Test / Temporal / Domain

Then: **Section 14 — Everything combined in one `sklearn.Pipeline`**

> Prottek step-e — *ki, keno, kivabe* + code + before/after proof (print/plot) thakbe,
> jate tumi nijer moto explain korte paro.

In [ ]:
# ---- Core libraries ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, random
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## 0. Raw Dataset (deliberately "dirty")

Ei function ta original XThreat dataset generator theke ektu expand kora hoyeche,
jate **prottek preprocessing problem intentionally thake**:

- kichu missing value (`geo_ip_reputation_score`)
- kichu **duplicate rows**
- kichu **negative / impossible value** (data-entry error simulate)
- ekta **categorical column** (`protocol`) — encoding demo-r jonno
- kichu **extreme outlier** row
- ekta **timestamp** column (`first_seen`) — temporal split demo-r jonno

In [ ]:
def generate_raw_dataset(n_ips=4000, attack_ratio=0.06, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    n_attack_ips = int(n_ips * attack_ratio)
    n_normal_ips = n_ips - n_attack_ips
    start_time = pd.Timestamp("2026-08-01")
    protocols = ["ssh", "rdp", "http", "ftp"]

    for i in range(n_normal_ips):
        n_events = rng.poisson(4) + 1
        offset_min = int(rng.integers(0, 14 * 24 * 60))
        first_seen = start_time + pd.Timedelta(minutes=offset_min)
        fail_count = rng.binomial(n_events, 0.08)
        success_count = n_events - fail_count
        rows.append(dict(
            ip_id=f"norm_{i}", first_seen=first_seen,
            login_attempts=n_events, failed_attempts=fail_count,
            success_attempts=success_count,
            unique_usernames_tried=rng.integers(1, 3),
            avg_seconds_between_attempts=float(rng.normal(1800, 600)) if n_events > 1 else 3600.0,
            time_span_minutes=float(max(rng.integers(60, 600), 1)),
            distinct_user_agents=rng.integers(1, 2),
            night_time_ratio=float(rng.beta(2, 6)),
            geo_ip_reputation_score=float(rng.beta(8, 2)),
            country_change_count=0,
            protocol=rng.choice(protocols, p=[0.5, 0.2, 0.25, 0.05]),
            is_attack=0
        ))

    for i in range(n_attack_ips):
        n_events = rng.integers(40, 400)
        offset_min = int(rng.integers(0, 14 * 24 * 60))
        first_seen = start_time + pd.Timedelta(minutes=offset_min)
        fail_ratio = rng.uniform(0.75, 0.98)
        fail_count = int(n_events * fail_ratio)
        success_count = n_events - fail_count
        credential_stuffing = rng.random() < 0.4
        unique_users = rng.integers(15, 60) if credential_stuffing else rng.integers(1, 3)
        rows.append(dict(
            ip_id=f"atk_{i}", first_seen=first_seen,
            login_attempts=n_events, failed_attempts=fail_count,
            success_attempts=success_count,
            unique_usernames_tried=unique_users,
            avg_seconds_between_attempts=float(rng.uniform(0.5, 8)),
            time_span_minutes=float(rng.uniform(2, 45)),
            distinct_user_agents=rng.integers(3, 12),
            night_time_ratio=float(rng.beta(6, 2)),
            geo_ip_reputation_score=float(rng.beta(2, 8)),
            country_change_count=rng.integers(0, 4),
            protocol=rng.choice(protocols, p=[0.55, 0.25, 0.1, 0.1]),
            is_attack=1
        ))

    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    df["failure_rate"] = df["failed_attempts"] / df["login_attempts"]

    # --- inject dirtiness on purpose ---
    # 1) missing values
    miss_idx = rng.choice(df.index, size=int(0.02 * len(df)), replace=False)
    df.loc[miss_idx, "geo_ip_reputation_score"] = np.nan

    # 2) duplicate rows (copy 15 random rows)
    dup_idx = rng.choice(df.index, size=15, replace=False)
    df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

    # 3) impossible negative values (data entry glitch)
    err_idx = rng.choice(df.index, size=5, replace=False)
    df.loc[err_idx, "failed_attempts"] = -1

    # 4) extreme outliers
    out_idx = rng.choice(df.index, size=6, replace=False)
    df.loc[out_idx, "login_attempts"] = df["login_attempts"].max() * rng.uniform(15, 30, size=6)

    return df.reset_index(drop=True)

df_raw = generate_raw_dataset()
print(df_raw.shape)
df_raw.head()

## 1. Missing Data

**Keno matter kore:** most ML algorithm NaN handle korte pare na (crash), r
naively drop/fill korle bias ba **leakage** aste pare.

**Ki dekhbo:**
- kon column-e koto % missing
- 3 ta strategy: (a) row drop, (b) median/mode impute, (c) missing-indicator flag

In [ ]:
missing_summary = df_raw.isna().sum()
missing_pct = (missing_summary / len(df_raw) * 100).round(2)
pd.DataFrame({"missing_count": missing_summary, "missing_%": missing_pct}).query("missing_count > 0")

In [ ]:
df_missing_demo = df_raw.copy()

# (a) drop rows with missing values -- only viable when missing % is tiny
df_dropped = df_missing_demo.dropna(subset=["geo_ip_reputation_score"])
print("Rows before drop:", len(df_missing_demo), "| after drop:", len(df_dropped))

# (b) median imputation (numeric) -- our chosen strategy going forward
median_val = df_missing_demo["geo_ip_reputation_score"].median()
df_missing_demo["geo_ip_reputation_score_imputed"] = df_missing_demo["geo_ip_reputation_score"].fillna(median_val)

# (c) missing-indicator flag -- lets the model learn "was this value missing?"
df_missing_demo["geo_ip_reputation_score_was_missing"] = df_missing_demo["geo_ip_reputation_score"].isna().astype(int)

print("Remaining NaNs after impute:", df_missing_demo["geo_ip_reputation_score_imputed"].isna().sum())
df_missing_demo[["geo_ip_reputation_score", "geo_ip_reputation_score_imputed",
                  "geo_ip_reputation_score_was_missing"]].head(8)

> ⚠️ **Note (preview of Step 13):** ei demo-te median gotoTOTAL data theke calculate kora hoyeche,
> shudhu shekhar jonno. Real pipeline-e amra eta **train set** theke calculate korbo, test set-e
> shudhu apply korbo — noile eta **leakage**.

## 2. Cleaning

**Ki thik kori:**
- duplicate rows
- impossible / invalid values (e.g. negative `failed_attempts`)
- data type consistency
- string/id column-er whitespace, casing

In [ ]:
df_clean = df_raw.copy()

print("Duplicate rows (excluding ip_id):",
      df_clean.drop(columns=["ip_id"]).duplicated().sum())

df_clean = df_clean.drop_duplicates(subset=[c for c in df_clean.columns if c != "ip_id"])
print("Shape after removing duplicates:", df_clean.shape)

print("\nRows with impossible negative failed_attempts:", (df_clean["failed_attempts"] < 0).sum())
df_clean = df_clean[df_clean["failed_attempts"] >= 0]
print("Shape after removing invalid rows:", df_clean.shape)

# id / string cleanup
df_clean["ip_id"] = df_clean["ip_id"].str.strip().str.lower()
df_clean["protocol"] = df_clean["protocol"].str.strip().str.lower()

# sanity: failed + success should equal login_attempts
mismatch = (df_clean["failed_attempts"] + df_clean["success_attempts"] != df_clean["login_attempts"]).sum()
print("Rows where failed+success != total attempts:", mismatch)

## 3. Outlier Handling

**Cyber-security specific caution:** normal data-e outlier "noise", kintu ei domain-e
extreme outlier **NIJEI attack signal** hote pare! Tai blindly delete kora bipojjonok.
Amra tai:
1. IQR method diye detect kori
2. Domain judgement: bhalo kore dekhi eta genuine attack naki data-entry glitch
3. Genuine glitch hole **cap (winsorize)** kori, delete na kore

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_bounds(df_clean["login_attempts"])
outliers = df_clean[(df_clean["login_attempts"] < low) | (df_clean["login_attempts"] > high)]
print(f"IQR bounds for login_attempts: [{low:.1f}, {high:.1f}]")
print("Outlier rows:", len(outliers))
print("Of which are labeled attack:", outliers["is_attack"].sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df_clean["login_attempts"], ax=axes[0])
axes[0].set_title("login_attempts (before capping)")

# cap extreme values at the 99.5th percentile (winsorize) instead of deleting
cap_value = df_clean["login_attempts"].quantile(0.995)
df_clean["login_attempts_capped"] = df_clean["login_attempts"].clip(upper=cap_value)

sns.boxplot(x=df_clean["login_attempts_capped"], ax=axes[1])
axes[1].set_title(f"login_attempts (capped at {cap_value:.0f})")
plt.tight_layout(); plt.show()

## 4. Sampling

Ei step-e ami **data-set thekey subset newar** general technique dekhabo
(imbalance-fix na — oita step 5-e).

- **Simple random sampling** — quick EDA / prototyping-er jonno
- **Stratified sampling** — class-ratio bojay rekhe subset newa (train/test split-er
  foundation, jeta amra step 13-e formally use korbo)

In [ ]:
# simple random sample (5% of data) -- useful for quick exploration on huge logs
sample_random = df_clean.sample(frac=0.05, random_state=SEED)
print("Random sample class ratio:\n", sample_random["is_attack"].value_counts(normalize=True))

# stratified sample -- keeps same attack:normal ratio as full data
from sklearn.model_selection import train_test_split
sample_strat, _ = train_test_split(
    df_clean, train_size=0.05, stratify=df_clean["is_attack"], random_state=SEED
)
print("\nStratified sample class ratio:\n", sample_strat["is_attack"].value_counts(normalize=True))
print("\nFull data class ratio:\n", df_clean["is_attack"].value_counts(normalize=True))

## 5. Imbalance Handling

Attack IP mate ~6% — heavily imbalanced. Naively train korle model "shob normal"
bole diye o 94% accuracy pabe, kintu attack e ekta o dhorbe na. Options:

1. **Class weights** — model-ke bole minority class-er mistake-ke beshi punish korte
2. **Random oversampling** — minority class-er row duplicate kore
3. **Random undersampling** — majority class kome ana
4. **SMOTE (concept)** — synthetic minority samples toiri kora (2 real minority
   point-er majhe interpolate kore) — from-scratch mini version niche dekhano hocche

In [ ]:
print("Class counts:\n", df_clean["is_attack"].value_counts())
print("\nClass %:\n", (df_clean["is_attack"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# work only on numeric feature columns for the demo
demo_cols = ["login_attempts_capped", "failed_attempts", "failure_rate", "avg_seconds_between_attempts"]
minority = df_clean[df_clean.is_attack == 1][demo_cols].reset_index(drop=True)
majority = df_clean[df_clean.is_attack == 0][demo_cols].reset_index(drop=True)
print("Minority (attack) rows:", len(minority), "| Majority (normal) rows:", len(majority))

# (1) random oversampling -- duplicate minority rows with replacement until balanced
rng = np.random.default_rng(SEED)
oversample_idx = rng.integers(0, len(minority), size=len(majority) - len(minority))
minority_oversampled = pd.concat([minority, minority.iloc[oversample_idx]], ignore_index=True)
print("After random oversampling -> minority rows:", len(minority_oversampled))

# (2) random undersampling -- drop majority rows down to minority size
majority_undersampled = majority.sample(n=len(minority), random_state=SEED)
print("After random undersampling -> majority rows:", len(majority_undersampled))

# (3) mini from-scratch SMOTE: pick a minority point + a random minority neighbour,
#     interpolate a new synthetic point between them
def manual_smote(minority_df, n_new, seed=SEED):
    rng = np.random.default_rng(seed)
    arr = minority_df.values
    n = len(arr)
    synthetic = []
    for _ in range(n_new):
        i, j = rng.integers(0, n, size=2)
        gap = rng.random()
        new_point = arr[i] + gap * (arr[j] - arr[i])
        synthetic.append(new_point)
    return pd.DataFrame(synthetic, columns=minority_df.columns)

n_needed = len(majority) - len(minority)
synthetic_minority = manual_smote(minority, n_needed)
minority_smote = pd.concat([minority, synthetic_minority], ignore_index=True)
print("After manual SMOTE -> minority rows:", len(minority_smote))

> Pipeline section-e (14) amra `imbalanced-learn`-er official `SMOTE` use korbo, kintu
> upor-er code-tai dekhay eta **actually kivabe kaj kore** — jate blindly library call
> na kore concept-ta bujhe use korte paro.

## 6. Transformation

Onek feature **skewed** (e.g. `avg_seconds_between_attempts`) — kichu model
(linear/distance-based) skewed data-e valo perform kore na. Solution: **log /
power transform** kore distribution ke normal-er kachakachi ana.

In [ ]:
from scipy.stats import skew

col = "avg_seconds_between_attempts"
raw_skew = skew(df_clean[col].dropna())
log_transformed = np.log1p(df_clean[col].clip(lower=0))
log_skew = skew(log_transformed)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean[col], bins=40, ax=axes[0])
axes[0].set_title(f"Before transform (skew={raw_skew:.2f})")
sns.histplot(log_transformed, bins=40, ax=axes[1])
axes[1].set_title(f"After log1p transform (skew={log_skew:.2f})")
plt.tight_layout(); plt.show()

## 7. Scaling

Feature gulor scale onek different (`avg_seconds_between_attempts` ~ thousands,
`night_time_ratio` ~ 0-1). Distance/gradient-based model-er jonno eta somoshsa.
3 ta common scaler compare kori:

- **StandardScaler** — mean 0, std 1 (default choice)
- **MinMaxScaler** — 0 to 1 range e squeeze
- **RobustScaler** — median/IQR base, outlier-resistant

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

num_col = "login_attempts_capped"
sample_vals = df_clean[[num_col]].copy()

scalers = {"StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler(), "RobustScaler": RobustScaler()}
fig, axes = plt.subplots(1, len(scalers) + 1, figsize=(16, 3.5))
sns.histplot(sample_vals[num_col], ax=axes[0], bins=30); axes[0].set_title("Original")

for ax, (name, scaler) in zip(axes[1:], scalers.items()):
    scaled = scaler.fit_transform(sample_vals)
    sns.histplot(scaled.ravel(), ax=ax, bins=30)
    ax.set_title(name)
plt.tight_layout(); plt.show()

> ⚠️ Preview of Step 13: scaler shobshomoy **`.fit()` shudhu train data-e**, tarpor
> `.transform()` train and test dutoyy-e — noile test-set-er info train-e "leak" hoy.

## 8. Encoding

`protocol` column (ssh/rdp/http/ftp) categorical — ML model number chai, string na.
Dekhbo:
- **One-Hot Encoding** — nominal category (order nei) jonno best
- **Label Encoding** — ordinal / tree-model-er jonno compact option

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

print(df_clean["protocol"].value_counts())

# One-hot encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_arr = ohe.fit_transform(df_clean[["protocol"]])
ohe_df = pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(["protocol"]), index=df_clean.index)
display(ohe_df.head())

# Label encoding (compact, fine for tree-based models)
le = LabelEncoder()
df_clean["protocol_label"] = le.fit_transform(df_clean["protocol"])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

## 9. Feature Engineering

Raw log theke shudhu count na, **behavioral signal** toiri kori — jeta attack vs
normal alada korte sobcheye beshi shahajjo kore:

- `attempts_per_minute` — burst-rate
- `burstiness_score` — koto fast pore pore attempt (1 / avg-gap)
- `username_diversity` — koyta different username try hoyeche (credential-stuffing signal)
- `failure_rate` — already present, but re-derived here after cleaning

In [ ]:
def engineer_features(data):
    data = data.copy()
    data["attempts_per_minute"] = data["login_attempts_capped"] / data["time_span_minutes"].clip(lower=1)
    data["burstiness_score"] = 1 / data["avg_seconds_between_attempts"].clip(lower=0.1)
    data["username_diversity"] = data["unique_usernames_tried"] / data["login_attempts_capped"].clip(lower=1)
    data["failure_rate"] = data["failed_attempts"] / data["login_attempts_capped"].clip(lower=1)
    return data

df_fe = engineer_features(df_clean)
new_feats = ["attempts_per_minute", "burstiness_score", "username_diversity", "failure_rate"]
df_fe.groupby("is_attack")[new_feats].mean().T

## 10. Feature Selection

Shob feature model-ke help nao korte pare — kichu redundant, kichu noise.
- **Correlation heatmap** — highly correlated (redundant) pair khuje ber kori
- **Correlation-with-target** — kon feature target predict korte beshi useful
- **`SelectKBest` (mutual information)** — statistically top-K feature bache

In [ ]:
CANDIDATE_FEATURES = [
    "login_attempts_capped", "failed_attempts", "success_attempts",
    "unique_usernames_tried", "avg_seconds_between_attempts", "time_span_minutes",
    "distinct_user_agents", "night_time_ratio", "geo_ip_reputation_score",
    "country_change_count", "failure_rate", "attempts_per_minute",
    "burstiness_score", "username_diversity", "protocol_label"
]
TARGET = "is_attack"

corr = df_fe[CANDIDATE_FEATURES + [TARGET]].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation matrix")
plt.tight_layout(); plt.show()

print("Correlation with target (sorted):")
print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False))

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

X_fs = df_fe[CANDIDATE_FEATURES].fillna(df_fe[CANDIDATE_FEATURES].median())
y_fs = df_fe[TARGET]

selector = SelectKBest(score_func=mutual_info_classif, k=8)
selector.fit(X_fs, y_fs)
scores = pd.Series(selector.scores_, index=CANDIDATE_FEATURES).sort_values(ascending=False)
print("Mutual-information feature ranking:")
print(scores)

SELECTED_FEATURES = scores.head(8).index.tolist()
print("\nTop 8 selected features:", SELECTED_FEATURES)

## 11. Simulation / Augmentation

Real world-e attack traffic collect kora costly/rare. Ei step-e amra kichu
**new synthetic attack variant** simulate kori — attacker-er behavior-e realistic
noise add kore — jate model rare/evolving attack pattern-eo generalize korte pare.

In [ ]:
def augment_attack_traffic(attack_df, n_new, noise_scale=0.08, seed=SEED):
    """Bootstrap an existing attack row + add small Gaussian jitter to numeric fields
       to simulate slightly-different, unseen attacker behaviour."""
    rng = np.random.default_rng(seed)
    numeric_cols = attack_df.select_dtypes(include=[np.number]).columns.drop("is_attack", errors="ignore")
    base_rows = attack_df.sample(n=n_new, replace=True, random_state=seed).reset_index(drop=True)
    for c in numeric_cols:
        jitter = rng.normal(0, noise_scale, size=n_new) * base_rows[c].std()
        base_rows[c] = (base_rows[c] + jitter).clip(lower=0)
    base_rows["ip_id"] = [f"sim_atk_{i}" for i in range(n_new)]
    return base_rows

attack_rows = df_fe[df_fe.is_attack == 1]
simulated_attacks = augment_attack_traffic(attack_rows, n_new=50)
print("Simulated new attack rows:", simulated_attacks.shape[0])

df_augmented = pd.concat([df_fe, simulated_attacks], ignore_index=True)
print("Dataset shape after augmentation:", df_augmented.shape)
print(df_augmented["is_attack"].value_counts())

## 12. Statistical Analysis

Feature engineering "chokh diye" na kore, **statistically proof** kori kon feature
gulo sotti-i attack vs normal-ke alada kore.
- **Descriptive stats** per class
- **Mann-Whitney U test** (non-parametric, distribution-free) — numeric feature
- **Chi-square test** — categorical feature (`protocol`) vs target

In [ ]:
from scipy.stats import mannwhitneyu, chi2_contingency

print("Descriptive stats by class (failure_rate):")
print(df_fe.groupby("is_attack")["failure_rate"].describe().T)

In [ ]:
results = []
for feat in ["failure_rate", "burstiness_score", "night_time_ratio", "geo_ip_reputation_score"]:
    normal_vals = df_fe.loc[df_fe.is_attack == 0, feat].dropna()
    attack_vals = df_fe.loc[df_fe.is_attack == 1, feat].dropna()
    stat, p = mannwhitneyu(normal_vals, attack_vals, alternative="two-sided")
    results.append({"feature": feat, "U_stat": stat, "p_value": p,
                     "significant_(p<0.05)": p < 0.05})
pd.DataFrame(results)

In [ ]:
# chi-square: is protocol choice statistically associated with being an attack?
contingency = pd.crosstab(df_fe["protocol"], df_fe["is_attack"])
chi2, p, dof, expected = chi2_contingency(contingency)
print(contingency)
print(f"\nChi2 = {chi2:.2f}, p-value = {p:.4f}  -> {'significant' if p < 0.05 else 'not significant'} association")

## 13. Extra Layer — Leakage / Train-Test / Temporal / Domain

Eta shobcheye beshi ignore kora hoy, kintu **shobcheye important** — evaluation
number ta "sotti bishwas jogyo" naki "misleadingly optimistic" eta ei layer decide kore.

**A. Data leakage —** kono statistic (mean/median/scaler/encoder) train set-er baire
kono kichu theke shikhle ba test-row-er info dekhe fit korle, evaluation fake-bhalo
dekhabe. **Rule: fit ONLY on train, transform train+test.**

**B. Train-test split (random vs stratified) —** class imbalance thakle random split
majhe majhe test set-e attack row kom pore jete pare — always **stratify** kora uchit।

**C. Temporal validation —** security data-te attacker tactic somoy-er sathe change
hoy (concept drift). Random shuffle split diye evaluate korle result **overly-optimistic**
lagbe. Real deployment simulate korte **time-based split** (train = purono somoy,
test = notun somoy) beshi honest.

**D. Domain sanity check —** emon kono feature ache ki jeta prediction-er somoy
*actually* available thakbe na (future info)? E.g. jodi `is_attack` label thekei
kono derived stat feature-e leak kore fela hoy, seta bug.

In [ ]:
# --- B: stratified split (correct) vs a naive non-stratified split (wrong) ---
from sklearn.model_selection import train_test_split

X_all = df_fe[CANDIDATE_FEATURES]
y_all = df_fe[TARGET]

_, _, _, y_test_naive = train_test_split(X_all, y_all, test_size=0.25, random_state=SEED)
_, _, _, y_test_strat = train_test_split(X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all)

print("Full-data attack ratio      :", round(y_all.mean(), 4))
print("Naive split test attack ratio :", round(y_test_naive.mean(), 4))
print("Stratified split test ratio   :", round(y_test_strat.mean(), 4))

In [ ]:
# --- C: temporal split, using the first_seen timestamp we generated in section 0 ---
df_time = df_fe.merge(df_raw[["ip_id", "first_seen"]], on="ip_id", how="left")
df_time_sorted = df_time.sort_values("first_seen")

split_point = int(len(df_time_sorted) * 0.75)
train_temporal = df_time_sorted.iloc[:split_point]
test_temporal = df_time_sorted.iloc[split_point:]

print("Temporal train window:", train_temporal["first_seen"].min(), "->", train_temporal["first_seen"].max())
print("Temporal test window :", test_temporal["first_seen"].min(), "->", test_temporal["first_seen"].max())
print("\nTrain attack ratio:", round(train_temporal["is_attack"].mean(), 4))
print("Test attack ratio  :", round(test_temporal["is_attack"].mean(), 4))

In [ ]:
# --- A: demonstrating leakage vs leakage-safe imputation/scaling side by side ---
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all
)

# WRONG (leaky): median computed on the FULL dataset (train+test mixed)
leaky_median = X_all.median()
X_test_leaky = X_test.fillna(leaky_median)

# CORRECT (safe): median computed on TRAIN ONLY
safe_median = X_train.median()
X_test_safe = X_test.fillna(safe_median)

diff = (leaky_median - safe_median).abs().sort_values(ascending=False)
print("How much the leaky vs safe median differs per feature (should be small but non-zero):")
print(diff.head())
print("\n-> In a real deployment gap can be much larger; always fit on train only.")

## 14. Everything Combined — `sklearn.Pipeline`

Ekhon shob step ek jaygay: `ColumnTransformer` (impute + scale + encode) →
`SelectKBest` (feature selection) → `SMOTE` (imbalance, train-fold-only via
`imblearn.pipeline.Pipeline`) → classifier. Ekta single `.fit()` / `.predict()` call-e
puro preprocessing + modeling hoye jabe, r leakage automatically avoid hobe karon
proti CV fold-e shudhu train-fold theke fit hobe.

> `imbalanced-learn` na thakle: `pip install imbalanced-learn`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, f1_score

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed -> run `pip install imbalanced-learn` for the SMOTE step")

In [ ]:
NUMERIC_FEATURES = [
    "login_attempts_capped", "failed_attempts", "success_attempts",
    "unique_usernames_tried", "avg_seconds_between_attempts", "time_span_minutes",
    "distinct_user_agents", "night_time_ratio", "geo_ip_reputation_score",
    "country_change_count", "failure_rate", "attempts_per_minute",
    "burstiness_score", "username_diversity",
]
CATEGORICAL_FEATURES = ["protocol"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "is_attack"

X = df_fe[ALL_FEATURES]
y = df_fe[TARGET]

# stratified split -- the only split that touches raw X/y before the pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Step 1+7+8 combined: impute -> scale (numeric) / impute -> one-hot (categorical)
numeric_transform = SkPipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_transform = SkPipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transform, NUMERIC_FEATURES),
    ("cat", categorical_transform, CATEGORICAL_FEATURES),
])

In [ ]:
if HAS_IMBLEARN:
    full_pipeline = ImbPipeline(steps=[
        ("preprocess", preprocess),                                  # steps 1,2,7,8
        ("select", SelectKBest(score_func=mutual_info_classif, k=10)),  # step 10
        ("smote", SMOTE(random_state=SEED)),                          # step 5
        ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
else:
    full_pipeline = SkPipeline(steps=[
        ("preprocess", preprocess),
        ("select", SelectKBest(score_func=mutual_info_classif, k=10)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ])

full_pipeline.fit(X_train, y_train)   # single call = imputation+scaling+encoding+selection+SMOTE+training

y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["normal", "attack"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))

### Why this matters (recap)

- `.fit(X_train, y_train)` **only ever sees the train fold** — imputer median,
  scaler mean/std, SMOTE synthetic points, feature-selection scores — sob train-only.
- `.predict(X_test)` just **transforms** using what was already learned — kono
  test-set info train-e leak hoy na.
- Cross-validation-e (e.g. `cross_val_score(full_pipeline, X, y, cv=5)`) use korle
  eita ekdom automatically prottek fold-e thik-thak hobe — eta-e pipeline use korar
  main benefit.
- Real deployment-e is temporal drift-er kotha mathay rekhe periodically **temporal
  split** (Step 13-C) diye o re-evaluate kora uchit।